# Routing

In [1]:
%matplotlib inline
from IPython.display import display
import pandas as pd

from dualde import load_example
from dualde.routing import run_information_transfer_pipeline

EXAMPLE, cfg = load_example()
OUT = EXAMPLE / "outputs"
scores = pd.read_csv(EXAMPLE / "scores" / "phenotype.csv")
coup_path = EXAMPLE / "couplings" / cfg["couplings_file"]
result = run_information_transfer_pipeline(
    all_merged_df=scores,
    couplings_file=str(coup_path),
    functional_residues_file=str(OUT / "functional_residues.csv"),
    active_site_ligands=list(cfg["ligand_names"]),
    activity_wt=cfg["activity_wt"],
    abundance_wt=cfg["abundance_wt"],
    score_col=cfg["routing_score_col"],
    low_score_percentile=cfg["low_score_percentile"],
    prob_threshold=cfg["prob_threshold"],
    n_paths=cfg["n_paths"],
    weighted=True,
    dist_cutoff=cfg["dist_cutoff"],
    functional_dist_cutoff=cfg["functional_dist_cutoff"],
    seq_len=cfg["seq_len"],
    verbose=False,
)
paths, pairs = result["paths_df"], result["pairs_df"]
paths.to_csv(OUT / "coupling_paths.csv", index=False)
pairs.to_csv(OUT / "coupling_pairs.csv", index=False)
long_paths = paths.loc[paths["n_edges"] > cfg["min_n_edges"]].copy()
print(len(paths), "paths", int((pairs["n_paths_found"] > 0).sum()), "pairs")
print(len(long_paths), "paths with more than", cfg["min_n_edges"], "edges")
display(long_paths.nsmallest(8, "n_edges")[["source", "target", "path_rank", "n_edges", "path_str"]])


611 paths 123 pairs
530 paths with more than 4 edges


,source,target,path_rank,n_edges,path_str
11,15,82,2,5,15 -> 17 -> 18 -> 97 -> 56 -> 82
16,15,91,2,5,15 -> 49 -> 92 -> 13 -> 12 -> 91
28,15,135,4,5,15 -> 49 -> 92 -> 94 -> 137 -> 135
35,16,17,1,5,16 -> 59 -> 50 -> 49 -> 15 -> 17
46,16,82,2,5,16 -> 59 -> 50 -> 14 -> 93 -> 82
51,16,91,2,5,16 -> 59 -> 50 -> 14 -> 12 -> 91
53,16,91,4,5,16 -> 59 -> 50 -> 14 -> 93 -> 91
55,16,94,1,5,16 -> 59 -> 50 -> 49 -> 92 -> 94


In [ ]:
long_paths = long_paths[long_paths["path_cost"] < 100]

Index(['source', 'target', 'path_rank', 'path', 'path_str', 'n_edges',
       'path_cost', 'path_coupling_score', 'min_edge_prob', 'mean_edge_prob',
       'edge_probs'],
      dtype='str')

In [9]:
# sort by path_cost
long_paths.sort_values("path_cost", ascending=True, inplace=True)
long_paths.head()

,source,target,path_rank,path,path_str,n_edges,path_cost,path_coupling_score,min_edge_prob,mean_edge_prob,edge_probs
260,49,91,1,"[49, 92, 13, 11, 12, 91]",49 -> 92 -> 13 -> 11 -> 12 -> 91,5,0.233904,-0.233904,0.847566,0.953219,"[0.9999459037854854, 0.9428562488171328, 0.999..."
435,94,91,1,"[94, 92, 13, 11, 12, 91]",94 -> 92 -> 13 -> 11 -> 12 -> 91,5,0.324605,-0.324605,0.847566,0.935079,"[0.9092442070264024, 0.9428562488171328, 0.999..."
365,90,91,1,"[90, 92, 13, 11, 12, 91]",90 -> 92 -> 13 -> 11 -> 12 -> 91,5,0.363391,-0.363391,0.847566,0.927322,"[0.8704582791347568, 0.9428562488171328, 0.999..."
330,88,91,1,"[88, 90, 92, 13, 11, 12, 91]",88 -> 90 -> 92 -> 13 -> 11 -> 12 -> 91,6,0.377111,-0.377111,0.847566,0.937148,"[0.986280424910474, 0.8704582791347568, 0.9428..."
540,136,135,1,"[136, 140, 83, 81, 94, 137, 135]",136 -> 140 -> 83 -> 81 -> 94 -> 137 -> 135,6,0.381018,-0.381018,0.805287,0.936497,"[0.998161205725059, 0.970693160862377, 0.99844..."


In [2]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

from dualde.path_tests import (
    result_record,
    test_path_vs_contact_decoys,
    test_path_vs_rsa_matched_contact_decoys,
    test_path_vs_rsa_matched_sets,
)
from dualde.structure_graph import build_contact_graph, residue_distance_map, residue_rsa_map

pdb_path = EXAMPLE / "pdb" / f"{cfg['pdb_id']}_cleaned.pdb"
dist = residue_distance_map(pdb_path, chain_id=cfg["chain"], atom="CA")
scored = set(scores["target_pos"].dropna().astype(int))
rsa_map = {
    int(pos): round(float(value), 3)
    for pos, value in residue_rsa_map(pdb_path, chain_id=cfg["chain"], cap_at_monomer=False).items()
    if int(pos) in scored
}
contact = build_contact_graph(dist, cutoff=cfg["contact_distance_cutoff"])
print(contact.number_of_nodes(), "residues", contact.number_of_edges(), "contacts", len(rsa_map), "RSA")

common = dict(
    df=scores,
    n_null=1000,
    seed=0,
    rsa_map=rsa_map,
    exclude_endpoints=True,
    statistic="both_low",
    alternative="greater",
    activity_wt=cfg["activity_wt"],
    abundance_wt=cfg["abundance_wt"],
    verbose=False,
)
rows = []
for record in long_paths.itertuples(index=False):
    path = [int(pos) for pos in record.path]
    tests = {
        "contact": test_path_vs_contact_decoys(
            path, dist_df=dist, cutoff=cfg["contact_distance_cutoff"], match_buriedness=False, **common
        ),
        "contact_rsa": test_path_vs_rsa_matched_contact_decoys(
            path, dist_df=dist, cutoff=cfg["contact_distance_cutoff"], rsa_tol=0.1, **common
        ),
        "rsa_sets": test_path_vs_rsa_matched_sets(path, rsa_tol=0.1, **common),
    }
    for res in tests.values():
        row = result_record(res)
        null = res["null_stats"]
        row["z"] = float((res["observed"] - null.mean()) / null.std())
        row["source"] = int(record.source)
        row["target"] = int(record.target)
        row["path_rank"] = int(record.path_rank)
        row["n_edges"] = int(record.n_edges)
        row["path"] = " -> ".join(map(str, path))
        rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "path_null_tests.csv", index=False)
display(summary.groupby("test")[["observed", "null_mean", "p_value", "z"]].describe())

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4), constrained_layout=True)
for ax, test in zip(axes, ["contact", "contact_rsa", "rsa_sets"]):
    vals = summary.loc[summary["test"] == test, "p_value"]
    ax.hist(vals, bins=30, color="0.75", edgecolor="white")
    ax.axvline(0.05, color="C3", lw=2)
    ax.set_title(f"{test}\n{int((vals < 0.05).sum())} / {len(vals)} p < 0.05")
    ax.set_xlabel("p")
    ax.set_ylabel("paths")
axes[0].legend(handles=[Line2D([0], [0], color="C3", lw=2, label="p = 0.05")], frameon=False)
fig.savefig(OUT / "path_null_tests.png", dpi=150)
plt.show()


156 residues 721 contacts 127 RSA


KeyboardInterrupt: 